In [ ]:
# ==============================================================================
# PCA Gene Expression Pipeline for .RDA Files (e.g., Golub Leukemia Dataset)
# ==============================================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Automatically install pyreadr if not present (for reading .rda/.RData files)
try:
    import pyreadr
except ImportError:
    print("Installing pyreadr to parse .rda file...")
    import subprocess
    subprocess.check_call(["pip", "install", "pyreadr", "-q"])
    import pyreadr

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

warnings.filterwarnings("ignore")
RANDOM_SEED = 42

# --- 1. SINGLE .RDA FILE LOADER ---
def load_rda_dataset():
    """
    Prompts the user to upload the .rda file and extracts
    expression data and labels automatically.
    """
    try:
        from google.colab import files
        print("Google Colab environment detected.")
        print("\nPlease upload your .rda file (e.g., leukemia_data_Golub99_3051.rda):")
        uploaded = files.upload()
        rda_filename = list(uploaded.keys())[0]

        print(f"Parsing R data archive: {rda_filename}...")
        result = pyreadr.read_r(rda_filename)

        # pyreadr returns a dictionary of pandas DataFrames stored in the .rda file
        print(f"Objects found inside RDA: {list(result.keys())}")

        # Extract dataframes dynamically
        dfs = list(result.values())
        if len(dfs) == 1:
            df = dfs[0]
            # If everything is in one dataframe, assume the last column or a specific name contains labels
            print(f"Loaded single dataframe with shape: {df.shape}")
            # Look for typical label column names, or default to the last column
            label_candidates = [col for col in df.columns if col.lower() in ['class', 'label', 'subtype', 'status', 'y']]
            if label_candidates:
                label_col = label_candidates[0]
            else:
                label_col = df.columns[-1]  # fallback to last column

            y_data = df[label_col]
            X_data = df.drop(columns=[label_col])
        else:
            # If multiple objects, look for a feature matrix and a label vector
            X_data, y_data = None, None
            for key, val in result.items():
                if val.shape[1] > 50:  # Large width usually means genes/features
                    X_data = val
                else:
                    y_data = val.iloc[:, 0] # First column of label object
            if X_data is None or y_data is None:
                X_data = dfs[0]
                y_data = dfs[1].iloc[:, 0]

        return X_data, y_data, f"Uploaded RDA Archive ({rda_filename})"

    except Exception as e:
        print(f"\nCould not parse uploaded file directly ({e}). Falling back to simulation stand-in.")
        return generate_alternative_transcriptomic_data()

def generate_alternative_transcriptomic_data(n_samples=600, n_genes=3051, seed=RANDOM_SEED):
    """Fallback simulation matching typical leukemia gene dimensions if upload fails."""
    rng = np.random.default_rng(seed)
    subtypes = ["ALL", "AML"]
    labels = rng.choice(subtypes, size=n_samples, p=[0.6, 0.4])
    latent_factors = rng.normal(0, 1, size=(n_samples, 20))
    for i, st in enumerate(subtypes):
        mask = (labels == st)
        latent_factors[mask, i*5:(i+1)*5] += 2.5
    loadings = rng.normal(0, 1, size=(n_genes, 20))
    noise = rng.normal(0, 0.8, size=(n_samples, n_genes))
    expression_values = np.clip(np.abs(latent_factors @ loadings.T + noise) * 10, 0, 1000)
    X_df = pd.DataFrame(expression_values, columns=[f"Gene_{i}" for i in range(n_genes)])
    y_ser = pd.Series(labels, name="Leukemia_Type")
    return X_df, y_ser, "Simulated Leukemia Stand-in (3,051 genes)"


# --- 2. MAIN EXECUTION PIPELINE ---
def run_pipeline():
    os.makedirs("output_results", exist_ok=True)

    # Load Data from RDA
    X_raw, y_raw, source_desc = load_rda_dataset()
    print(f"\nDataset Source: {source_desc}")
    print(f"Original Shape -> Samples (n): {X_raw.shape[0]} | Features/Genes (p): {X_raw.shape[1]}")

    # Clean zero-variance genes
    var_check = X_raw.var(axis=0) > 0
    X_clean = X_raw.loc[:, var_check]
    print(f"Features after removing zero-variance columns: {X_clean.shape[1]}")

    # Encode Labels
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y_raw.astype(str))
    class_names = list(encoder.classes_)
    print(f"Target Classes Discovered: {class_names}")

    # Stratified Train-Test Split (PREVENTS DATA LEAKAGE)
    X_arr = X_clean.values.astype(np.float64)
    X_train, X_test, y_train, y_test = train_test_split(
        X_arr, y_encoded, test_size=0.2, stratify=y_encoded, random_state=RANDOM_SEED
    )

    # Standardize Training Data Only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- 3. PCA DIMENSIONALITY REDUCTION ---
    print("\nFitting PCA on training partitions to eliminate high-dimensional collinearity...")
    pca_full = PCA(svd_solver="full").fit(X_train_scaled)
    cum_variance = np.cumsum(pca_full.explained_variance_ratio_)

    # Target 90% cumulative variance threshold
    k_components = int(np.searchsorted(cum_variance, 0.90) + 1)
    print(f"Selected Principal Components (k): {k_components} (Captures {cum_variance[k_components-1]*100:.2f}% cumulative variance)")
    print(f"Compression Factor: {X_clean.shape[1] / k_components:.1f}x reduction")

    # Transform train and test sets into orthogonal subspace
    pca_transformer = PCA(n_components=k_components, svd_solver="full")
    Z_train = pca_transformer.fit_transform(X_train_scaled)
    Z_test = pca_transformer.transform(X_test_scaled)

    # Verify Orthogonality
    pc_corr = np.corrcoef(Z_train, rowvar=False)
    off_diag_max = np.abs(pc_corr[np.triu_indices(k_components, 1)]).max()
    print(f"Orthogonality check: Max off-diagonal correlation = {off_diag_max:.2e} (Collinearity eliminated)")

    # --- 4. DOWNSTREAM CLASSIFICATION ---
    print("\nEvaluating downstream classifiers (Raw Genes vs. PCA Features)...")
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

    classifiers = {
        "Logistic Regression": LogisticRegression(max_iter=1500),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
    }

    comparison_records = []
    for clf_name, model in classifiers.items():
        # Raw Pipeline
        pipe_raw = Pipeline([("scaler", StandardScaler()), ("clf", model)])
        raw_scores = cross_val_score(pipe_raw, X_train, y_train, cv=cv_strategy, scoring="accuracy")

        # PCA Pipeline
        pipe_pca = Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=k_components)),
            ("clf", model)
        ])
        pca_scores = cross_val_score(pipe_pca, X_train, y_train, cv=cv_strategy, scoring="accuracy")

        comparison_records.append({
            "Classifier": clf_name,
            "Raw Genes CV Accuracy": f"{raw_scores.mean():.4f} (+/- {raw_scores.std():.4f})",
            "PCA Features CV Accuracy": f"{pca_scores.mean():.4f} (+/- {pca_scores.std():.4f})"
        })
        print(f"-> {clf_name} | Raw CV: {raw_scores.mean():.4f} | PCA CV: {pca_scores.mean():.4f}")

    results_df = pd.DataFrame(comparison_records)
    print("\n" + "="*60)
    print("MODEL COMPARISON SUMMARY")
    print("="*60)
    display(results_df) if 'display' in globals() else print(results_df)

    # --- 5. DISEASE SUBTYPING (UNSUPERVISED CLUSTERING) ---
    print("\nExecuting unsupervised K-Means disease subtyping in PCA subspace...")
    n_clusters = len(class_names)

    kmeans = KMeans(n_clusters=n_clusters, n_init=25, random_state=RANDOM_SEED)
    cluster_labels = kmeans.fit_predict(Z_train)

    ari = adjusted_rand_score(y_train, cluster_labels)
    sil = silhouette_score(Z_train, cluster_labels)

    print(f"-> Subtyping Adjusted Rand Index (ARI): {ari:.4f}")
    print(f"-> Subtyping Silhouette Score: {sil:.4f}")

    # --- 6. PLOTTING VISUALIZATIONS ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
    fig.patch.set_facecolor("#fcfcfb")

    # Scree Plot
    axes[0].bar(np.arange(1, min(51, len(pca_full.explained_variance_ratio_)+1)),
                pca_full.explained_variance_ratio_[:50] * 100, color="#2a78d6", alpha=0.8)
    axes[0].set_title("Scree Plot (Top Principal Components)", fontweight="bold", fontsize=11)
    axes[0].set_xlabel("Principal Component")
    axes[0].set_ylabel("Variance Explained (%)")
    axes[0].grid(True, linestyle="--", alpha=0.5)

    # 2D PC Scatter Subtyping View
    scatter = axes[1].scatter(Z_train[:, 0], Z_train[:, 1], c=y_train, cmap="viridis", s=25, alpha=0.8)
    axes[1].set_title("Patient Samples Projected on PC1 vs PC2", fontweight="bold", fontsize=11)
    axes[1].set_xlabel(f"PC1 ({pca_full.explained_variance_ratio_[0]*100:.1f}%)")
    axes[1].set_ylabel(f"PC2 ({pca_full.explained_variance_ratio_[1]*100:.1f}%)")
    axes[1].grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig("output_results/pipeline_visualizations.png")
    plt.show()
    print("\nPipeline execution completed successfully! Figures saved to 'output_results/'.")

if __name__ == "__main__":
    run_pipeline()

Google Colab environment detected.

Please upload your .rda file (e.g., leukemia_data_Golub99_3051.rda):


KeyboardInterrupt: 